In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [2]:
# 1. Load Data
df = pd.read_csv('mental_health_score.csv')

In [3]:
# 2. Drop Duplicate Rows
df = df.drop_duplicates()

# 3. Fix Invalid Data Anomalies
# Physical activity hours cannot be negative; clip lower bound at 0
df['Physical_Activity_Hours'] = df['Physical_Activity_Hours'].clip(lower=0.0)

In [4]:
df['Country'].value_counts()[df['Country'].value_counts() >= 50]

Country
Other          1879
India           389
USA             354
Canada          230
Australia       198
UK              185
Germany         136
Mexico           94
Turkey           94
France           87
Spain            83
Ireland          81
Denmark          77
Japan            77
Switzerland      73
Nepal            72
Italy            67
Russia           66
Sri Lanka        59
Maldives         57
Pakistan         53
Bangladesh       53
Poland           50
Name: count, dtype: int64

In [5]:
# 4. Consolidate High-Cardinality Categorical Columns
# Group infrequent countries (less than 1% representation) into 'Other'
top_countries = df['Country'].value_counts()[df['Country'].value_counts() >= 50].index
df['Country'] = df['Country'].apply(lambda x: x if x in top_countries[:10] else 'Other')
top_countries
df['Country'].value_counts()

Country
Other        3231
India         389
USA           354
Canada        230
Australia     198
UK            185
Germany       136
Mexico         94
Turkey         94
France         87
Name: count, dtype: int64

In [6]:

# 5. Define Feature Sets & Target
X = df.drop(columns=['Mental_Health_Score'])
y = df['Mental_Health_Score']




In [7]:
# Import XGBoost (or LightGBM)
from xgboost import XGBRegressor

In [31]:

# 1. Define Feature Names & Category Orders
numeric_features = [
    'Age', 'Avg_Daily_Usage_Hours', 'Daily_Unlocks', 
    'Study_Hours', 'Physical_Activity_Hours', 'Sleep_Hours_Per_Night'
]

academic_order = ['High School', 'Undergraduate', 'Graduate']
stress_order = ['Low', 'Medium', 'High', 'Very High']

nominal_features = ['Gender', 'Country', 'Most_Used_Platform', 'Purpose_Of_Use']

# 2. Construct Feature Transformers
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

academic_transformer = Pipeline(steps=[
    ('ordinal_academic', OrdinalEncoder(categories=[academic_order]))
])

stress_transformer = Pipeline(steps=[
    ('ordinal_stress', OrdinalEncoder(categories=[stress_order]))
])

nominal_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(sparse_output=False, handle_unknown='infrequent_if_exist'))
])

# 3. Combine into a Single ColumnTransformer Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('acad', academic_transformer, ['Academic_Level']),
        ('stress', stress_transformer, ['Stress_Level']),
        ('nom', nominal_transformer, nominal_features)
    ]
)

# 4. Build End-to-End Pipeline with Model
# Swap XGBRegressor with LGBMRegressor() if using LightGBM
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(
        n_estimators=500,        # Increased from 100 to give learning rate enough depth
        learning_rate=0.05,      # Keeps updates stable
        max_depth=6,             # Slightly increased capacity (default is 6)
        subsample=0.8,           # Prevents overfitting by using 80% data per tree
        colsample_bytree=0.8,    # Uses 80% features per tree
        random_state=42
    ))
])

# 5. Fit & Predict Directly on Raw Data Splitted Sets
# Assuming X and y are already extracted from your dataframe
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the entire pipeline (preprocessing + XGBoost fitting)
full_pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('acad', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [9]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


In [32]:
# Predict continuous scores
# y_pred = full_pipeline.predict(X_test)
y_train_pred = full_pipeline.predict(X_train)
y_test_pred  = full_pipeline.predict(X_test)

print(f"Training R² Score : {r2_score(y_train, y_train_pred):.4f}")
print(f"Testing R² Score  : {r2_score(y_test, y_test_pred):.4f}")
print(f"MAE (Test)        : {mean_absolute_error(y_test, y_test_pred):.4f}")

# # 1. Regression Metrics
# rmse = np.sqrt(mean_squared_error(y_test, y_pred))
# mae = mean_absolute_error(y_test, y_pred)
# r2 = r2_score(y_test, y_pred)

# # 2. Percentage Accuracy Within Error Margin (e.g., within +- 0.5 points)
# tolerance = 0.5
# accuracy_within_margin = np.mean(np.abs(y_test - y_pred) <= tolerance) * 100

# print(f"XGBoost Regression Evaluation:")
# print(f"  R² Score                   : {r2:.4f} ({r2 * 100:.2f}% variance explained)")
# print(f"  RMSE                       : {rmse:.4f}")
# print(f"  MAE                        : {mae:.4f}")
# print(f"  Accuracy (within ±{tolerance}) : {accuracy_within_margin:.2f}%")

Training R² Score : 0.9686
Testing R² Score  : 0.8869
MAE (Test)        : 0.3324


In [11]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_validate

In [12]:
# 1. Build End-to-End Pipeline with Linear Regression
lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

# 2. Define 5-Fold Cross-Validation Splitter
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# 3. Define Metrics to Evaluate Across Folds
scoring_metrics = {
    'r2': 'r2',
    'mae': 'neg_mean_absolute_error'
}

# 4. Perform 5-Fold Cross-Validation on Training Data (X_train)
cv_results = cross_validate(
    lr_pipeline, 
    X_train, 
    y_train, 
    cv=kf, 
    scoring=scoring_metrics,
    return_train_score=True
)

# Extract Mean Scores Across the 5 Folds
cv_r2_training = cv_results['train_r2'].mean()
cv_r2_validation = cv_results['test_r2'].mean()
cv_mae_validation = -cv_results['test_mae'].mean()  # Convert back from negative MAE

# 5. Fit the Model on Full X_train and Evaluate on Untouched X_test (Final Benchmark)
lr_pipeline.fit(X_train, y_train)


,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('acad', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [13]:

lr_preds_test = lr_pipeline.predict(X_test)

holdout_r2 = r2_score(y_test, lr_preds_test)
holdout_mae = mean_absolute_error(y_test, lr_preds_test)
holdout_rmse = np.sqrt(mean_squared_error(y_test, lr_preds_test))

# 6. Print Comprehensive Results
print("=" * 60)
print("  LINEAR REGRESSION — 5-FOLD CV & HOLDOUT RESULTS")
print("=" * 60)
print(f"5-Fold CV Train R² (Mean)     : {cv_r2_training:.4f} ({cv_r2_training * 100:.2f}%)")
print(f"5-Fold CV Val R² (Mean)       : {cv_r2_validation:.4f} ({cv_r2_validation * 100:.2f}%)")
print(f"5-Fold CV Val MAE (Mean)      : {cv_mae_validation:.4f}")
print("-" * 60)
print(f"Holdout Test R² Score        : {holdout_r2:.4f} ({holdout_r2 * 100:.2f}%)")
print(f"Holdout Test MAE              : {holdout_mae:.4f}")
print(f"Holdout Test RMSE             : {holdout_rmse:.4f}")
print("=" * 60)

  LINEAR REGRESSION — 5-FOLD CV & HOLDOUT RESULTS
5-Fold CV Train R² (Mean)     : 0.7271 (72.71%)
5-Fold CV Val R² (Mean)       : 0.7216 (72.16%)
5-Fold CV Val MAE (Mean)      : 0.5262
------------------------------------------------------------
Holdout Test R² Score        : 0.7436 (74.36%)
Holdout Test MAE              : 0.5328
Holdout Test RMSE             : 0.6766


In [14]:

from sklearn.ensemble import RandomForestRegressor

In [15]:
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('random forest', RandomForestRegressor(random_state=42))
])

rf_pipeline.fit(X_train, y_train)


,steps,"[('preprocessor', ...), ('random forest', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('acad', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [16]:
rf_preds          = rf_pipeline.predict(X_test)
rf_preds_training = rf_pipeline.predict(X_train)

rf_r2_testing  = r2_score(y_test, rf_preds)
rf_r2_training = r2_score(y_train, rf_preds_training)
rf_mae         = mean_absolute_error(y_test, rf_preds)

print(f"Accuracy of Training {rf_r2_training}")
print(f"Accuracy of Testing {rf_r2_testing}")
print(f"MAE : {rf_mae}")

Accuracy of Training 0.9825052515748018
Accuracy of Testing 0.8892585688096389
MAE : 0.3290296666666667


In [17]:
import joblib

# ── 1. SAVE THE TRAINED PIPELINE LOCALLY ─────────────────────────────────────
# Assuming 'full_pipeline' or 'lr_pipeline' is already fitted on your data
model_filename = "mental_health_random_forest_pipeline.pkl"

joblib.dump(rf_pipeline, model_filename)
print(f"✅ Model pipeline saved locally to: {model_filename}")

✅ Model pipeline saved locally to: mental_health_random_forest_pipeline.pkl


In [18]:
from sklearn.neural_network import MLPRegressor

In [19]:
# 1. Define the Neural Network Pipeline
# full_pipeline = Pipeline(steps=[
#     ('preprocessor', preprocessor),
#     ('regressor', MLPRegressor(
#         hidden_layer_sizes=(64, 32), # 2 hidden layers with 64 and 32 neurons
#         activation='relu',           # Activation function
#         solver='adam',               # Optimizer
#         max_iter=300,                # Number of epochs
#         random_state=42
#     ))
# ])
# full_pipeline = Pipeline(steps=[
#     ('preprocessor', preprocessor),
#     ('regressor', MLPRegressor(
#         hidden_layer_sizes=(64, 32),
#         activation='relu',
#         solver='adam',
#         alpha=0.01,             # Increased L2 regularization penalty (default is 0.0001)
#         early_stopping=True,    # Stops training when validation score stops improving
#         n_iter_no_change=15,    # Number of epochs to wait for improvement
#         validation_fraction=0.1,
#         random_state=42,
#         max_iter=500
#     ))
# ])

full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', MLPRegressor(
        hidden_layer_sizes=(128, 64), # Increased width from (64, 32)
        activation='relu',
        solver='adam',
        alpha=0.001,                   # Moderate regularization (default is 0.0001)
        learning_rate_init=0.001,
        early_stopping=True,
        n_iter_no_change=20,
        random_state=42,
        max_iter=500
    ))
])
# 2. Fit & Predict Directly on Raw Data Split Sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the entire pipeline (preprocessing + Neural Network fitting)
full_pipeline.fit(X_train, y_train)


,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('acad', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [20]:

# Generate predictions
nn_preds_training = full_pipeline.predict(X_train)
nn_preds = full_pipeline.predict(X_test)

# Evaluate metrics
nn_r2_testing = r2_score(y_test, nn_preds)
nn_r2_training = r2_score(y_train, nn_preds_training)
nn_mae = mean_absolute_error(y_test, nn_preds)

print(f"Accuracy of Training {nn_r2_training}")
print(f"Accuracy of Testing {nn_r2_testing}")
print(f"MAE : {nn_mae}")

Accuracy of Training 0.9112906604462641
Accuracy of Testing 0.8483266975421065
MAE : 0.38072034148834927


In [21]:
from sklearn.ensemble import RandomForestRegressor, VotingRegressor

In [22]:
nn_model = MLPRegressor(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    alpha=0.001,
    early_stopping=True,
    random_state=42
)

rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

# 2. Ensemble blending Neural Network + Random Forest
ensemble_model = VotingRegressor([
    ('nn', nn_model),
    ('rf', rf_model)
])

# 3. Create full pipeline
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', ensemble_model)
])

# 4. Fit and predict directly on split data
full_pipeline.fit(X_train, y_train)

c:\python 3.12\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('acad', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [23]:
ensemble_preds_training = full_pipeline.predict(X_train)
ensemble_preds = full_pipeline.predict(X_test)

# 5. Evaluate metrics
ensemble_r2_training = r2_score(y_train, ensemble_preds_training)
ensemble_r2_testing = r2_score(y_test, ensemble_preds)
ensemble_mae = mean_absolute_error(y_test, ensemble_preds)

print(f"Accuracy of Training {ensemble_r2_training}")
print(f"Accuracy of Testing {ensemble_r2_testing}")
print(f"MAE : {ensemble_mae}")

Accuracy of Training 0.9068155757410143
Accuracy of Testing 0.8599811941055816
MAE : 0.3767053225154393


In [24]:
# Improved Model 1: Constraining Model Complexity
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('random forest', RandomForestRegressor(
        n_estimators=100,
        max_depth=10,             # Prevents trees from growing infinitely deep
        min_samples_split=5,      # Requires at least 5 samples to split an internal node
        min_samples_leaf=2,       # Requires at least 2 samples to form a leaf node
        max_features='sqrt',      # Uses a random subset of features for each split (adds regularization)
        random_state=42
    ))
])

rf_pipeline.fit(X_train, y_train)


,steps,"[('preprocessor', ...), ('random forest', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('acad', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [25]:

rf_preds_training = rf_pipeline.predict(X_train)
rf_preds          = rf_pipeline.predict(X_test)

rf_r2_training = r2_score(y_train, rf_preds_training)
rf_r2_testing  = r2_score(y_test, rf_preds)
rf_mae         = mean_absolute_error(y_test, rf_preds)

print(f"Accuracy of Training {rf_r2_training}")
print(f"Accuracy of Testing {rf_r2_testing}")
print(f"MAE : {rf_mae}")

Accuracy of Training 0.866458825472275
Accuracy of Testing 0.833959164070004
MAE : 0.4199618396955837


In [26]:
from sklearn.model_selection import RandomizedSearchCV

In [27]:
# 1. Define the base pipeline (same as your original)
base_rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('random forest', RandomForestRegressor(random_state=42))
])

# 2. Define a grid of anti-overfitting parameters to test
param_grid = {
    'random forest__n_estimators': [100, 200],
    'random forest__max_depth': [5, 10, 15, 20],
    'random forest__min_samples_split': [2, 5, 10],
    'random forest__min_samples_leaf': [1, 2, 4],
    'random forest__max_features': [1.0, 'sqrt', 'log2']
}

# 3. Set up K-Fold Cross-Validation search
# cv=5 means 5-fold cross validation. It will train on 4 parts, test on 1 part, and rotate.
rf_cv = RandomizedSearchCV(
    estimator=base_rf_pipeline,
    param_distributions=param_grid,
    n_iter=15,          # Tests 15 random combinations from the grid
    cv=5,               # 5-fold cross-validation
    scoring='r2', 
    n_jobs=-1,          # Uses all CPU cores for faster processing
    random_state=42
)

# 4. Fit the search on training data
# This will take a little longer as it is training multiple models behind the scenes
rf_cv.fit(X_train, y_train)


,estimator,Pipeline(step...m_state=42))])
,param_distributions,"{'random forest__max_depth': [5, 10, ...], 'random forest__max_features': [1.0, 'sqrt', ...], 'random forest__min_samples_leaf': [1, 2, ...], 'random forest__min_samples_split': [2, 5, ...], ...}"
,n_iter,15
,scoring,'r2'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [29]:

# 5. Predict using the absolute best model it found
rf_preds_training = rf_cv.predict(X_train)
rf_preds          = rf_cv.predict(X_test)

# 6. Evaluate metrics
rf_r2_training = r2_score(y_train, rf_preds_training)
rf_r2_testing  = r2_score(y_test, rf_preds)
rf_mae         = mean_absolute_error(y_test, rf_preds)

print(f"Best Parameters found: {rf_cv.best_params_}")
print(f"Accuracy of Training {rf_r2_training}")
print(f"Accuracy of Testing {rf_r2_testing}")
print(f"MAE : {rf_mae}")
print(rf_cv.best_params_)

Best Parameters found: {'random forest__n_estimators': 200, 'random forest__min_samples_split': 5, 'random forest__min_samples_leaf': 1, 'random forest__max_features': 'log2', 'random forest__max_depth': 20}
Accuracy of Training 0.9588816628798661
Accuracy of Testing 0.8858474112608022
MAE : 0.335746919781588
{'random forest__n_estimators': 200, 'random forest__min_samples_split': 5, 'random forest__min_samples_leaf': 1, 'random forest__max_features': 'log2', 'random forest__max_depth': 20}


In [35]:
from sklearn.model_selection import GridSearchCV

In [38]:
# 1. Base Pipeline
xgb_base_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(random_state=42))
])

# 2. Targeted Hyperparameter Grid for GridSearchCV
# We lock in the good parameters and only test the strict constraints
xgb_param_grid_strict = {
    'regressor__n_estimators': [300, 500],             # Keep enough trees
    'regressor__learning_rate': [0.05, 0.1],           # Slow down the learning
    'regressor__max_depth': [4, 5, 6],                 # Stop trees from growing too deep (was 7)
    'regressor__gamma': [1, 3, 5],                     # Brutally prune useless branches (was 0)
    'regressor__subsample': [0.8],                     # Locked in from previous best
    'regressor__colsample_bytree': [0.8],              # Locked in from previous best
    'regressor__reg_alpha': [0.1, 0.5],                # L1 Regularization to drop useless features
    'regressor__reg_lambda': [2, 3]                    # L2 Regularization to shrink weights
}

# 3. Initialize GridSearchCV (Exhaustive search over the specified grid)
xgb_grid_search = GridSearchCV(
    estimator=xgb_base_pipeline,
    param_grid=xgb_param_grid_strict,
    cv=5,                         # 5-fold cross-validation
    scoring='r2',                 # Optimize for R-squared
    n_jobs=-1,                    # Use all available CPU cores
    verbose=1                     # Show progress
)
print("Starting targeted GridSearchCV for XGBoost...")
xgb_grid_search.fit(X_train, y_train)

Starting targeted GridSearchCV for XGBoost...
Fitting 5 folds for each of 144 candidates, totalling 720 fits


,estimator,"Pipeline(step...=None, ...))])"
,param_grid,"{'regressor__colsample_bytree': [0.8], 'regressor__gamma': [1, 3, ...], 'regressor__learning_rate': [0.05, 0.1], 'regressor__max_depth': [4, 5, ...], ...}"
,scoring,'r2'
,n_jobs,-1
,refit,True
,cv,5
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...), ('acad', ...), ...]"


In [39]:
best_xgb_model = xgb_grid_search.best_estimator_
# 5. Evaluate the Best Model
y_train_pred = best_xgb_model.predict(X_train)
y_test_pred = best_xgb_model.predict(X_test)

# Metrics Calculation
r2_train = r2_score(y_train, y_train_pred)
r2_test = r2_score(y_test, y_test_pred)
mae_test = mean_absolute_error(y_test, y_test_pred)

print("=" * 50)
print("      Final Tuned XGBoost Evaluation")
print("=" * 50)
print(f"Best Parameters Found:\n{xgb_grid_search.best_params_}\n")
print(f"Training R² Score  : {r2_train:.4f} ({r2_train * 100:.2f}%)")
print(f"Testing R² Score   : {r2_test:.4f} ({r2_test * 100:.2f}%)")
print(f"Train/Test Gap     : {(r2_train - r2_test) * 100:.2f}%")
print(f"MAE (Test)         : {mae_test:.4f}")
print("=" * 50)

      Final Tuned XGBoost Evaluation
Best Parameters Found:
{'regressor__colsample_bytree': 0.8, 'regressor__gamma': 1, 'regressor__learning_rate': 0.05, 'regressor__max_depth': 6, 'regressor__n_estimators': 500, 'regressor__reg_alpha': 0.1, 'regressor__reg_lambda': 2, 'regressor__subsample': 0.8}

Training R² Score  : 0.8658 (86.58%)
Testing R² Score   : 0.8333 (83.33%)
Train/Test Gap     : 3.25%
MAE (Test)         : 0.4236


In [42]:
# The "Goldilocks" XGBoost Model
final_xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(
        n_estimators=600,        # Increased from 500 to give it slightly more time to learn
        learning_rate=0.1,       # Kept at 0.1 for stable, consistent learning
        max_depth=7,             # Increased from 6. The 89.72% model proved your data NEEDS depth 7.
        gamma=0.2,               # Slightly increased to prune branches that don't add real value
        subsample=0.8,           # Locked in
        colsample_bytree=0.8,    # Locked in
        reg_alpha=0.5,           # Increased L1 (Lasso) from 0.1 to 0.5 to ignore noisy features
        reg_lambda=3,            # Increased L2 (Ridge) from 2 to 3 to keep weights small despite depth 7
        random_state=42
    ))
])

# Fit and Evaluate
final_xgb_pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('acad', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [43]:
y_train_pred = final_xgb_pipeline.predict(X_train)
y_test_pred = final_xgb_pipeline.predict(X_test)

r2_train = r2_score(y_train, y_train_pred)
r2_test = r2_score(y_test, y_test_pred)
mae_test = mean_absolute_error(y_test, y_test_pred)

print("=" * 50)
print("      The Final 'Goldilocks' XGBoost")
print("=" * 50)
print(f"Training R² Score  : {r2_train:.4f} ({r2_train * 100:.2f}%)")
print(f"Testing R² Score   : {r2_test:.4f} ({r2_test * 100:.2f}%)")
print(f"Train/Test Gap     : {(r2_train - r2_test) * 100:.2f}%")
print(f"MAE (Test)         : {mae_test:.4f}")
print("=" * 50)

      The Final 'Goldilocks' XGBoost
Training R² Score  : 0.9418 (94.18%)
Testing R² Score   : 0.8729 (87.29%)
Train/Test Gap     : 6.89%
MAE (Test)         : 0.3624


In [44]:
best_rf = RandomForestRegressor(
    n_estimators=200,         # (Update with your actual best RF params)
    max_depth=15,             # (Update with your actual best RF params)
    min_samples_split=2,      # (Update with your actual best RF params)
    min_samples_leaf=1,       # (Update with your actual best RF params)
    max_features='sqrt',      # (Update with your actual best RF params)
    random_state=42
)

# 2. Your Best XGBoost (The "Goldilocks" version that scored 88.15%)
best_xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.1,       
    max_depth=6,             
    gamma=0.1,               
    subsample=0.8,           
    colsample_bytree=0.8,    
    reg_alpha=0.1,           
    reg_lambda=2,            
    random_state=42
)

# 3. Create the Voting Ensemble
ensemble_model = VotingRegressor(estimators=[
    ('rf', best_rf),
    ('xgb', best_xgb)
])

# 4. Build the final pipeline
final_ensemble_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', ensemble_model)
])

# 5. Fit and Predict
final_ensemble_pipeline.fit(X_train, y_train)


,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('acad', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [45]:

y_train_pred = final_ensemble_pipeline.predict(X_train)
y_test_pred = final_ensemble_pipeline.predict(X_test)

# 6. Evaluate
r2_train = r2_score(y_train, y_train_pred)
r2_test = r2_score(y_test, y_test_pred)
mae_test = mean_absolute_error(y_test, y_test_pred)

print("=" * 50)
print("      Ultimate Voting Ensemble Evaluation")
print("=" * 50)
print(f"Training R² Score  : {r2_train:.4f} ({r2_train * 100:.2f}%)")
print(f"Testing R² Score   : {r2_test:.4f} ({r2_test * 100:.2f}%)")
print(f"Train/Test Gap     : {(r2_train - r2_test) * 100:.2f}%")
print(f"MAE (Test)         : {mae_test:.4f}")
print("=" * 50)

      Ultimate Voting Ensemble Evaluation
Training R² Score  : 0.9642 (96.42%)
Testing R² Score   : 0.8878 (88.78%)
Train/Test Gap     : 7.63%
MAE (Test)         : 0.3345
